# **Exploratory Data Analysis**

## Objectives

* Test the five hypotheses set out in the README against the cleaned data.
* Answer BR2: establish which customer attributes actually separate churned from
  retained customers, and which only appear to.
* Produce the ranking that notebook 04 turns into charts.

## Inputs

* `outputs/datasets/cleaned/bank_churners_cleaned.csv`

## Outputs

* `outputs/reports/variable_ranking.csv` - every variable scored on a common
  footing, used by notebook 04 and by the README conclusions.

## A note on method

This notebook deals in numbers; the charts live in notebook 04. Every comparison
is made as a **rate within each group**, never as a count, for the reason given
in notebook 01: the file is 5:1 imbalanced, so counts describe the sample size
rather than the behaviour.

Where a difference is claimed, it is backed by a significance test. With 10,127
rows almost anything reaches significance, so an effect size is reported
alongside every p-value, and the effect size is what the conclusions lean on.

---

# Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main\\jupyter_notebooks'

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main'

---

# Section 1: Load the cleaned data

CSV does not carry the ordered categories set in notebook 02, so they are
reapplied here rather than assumed.

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df = pd.read_csv('outputs/datasets/cleaned/bank_churners_cleaned.csv')

education_order = ['Uneducated', 'High School', 'College',
                   'Graduate', 'Post-Graduate', 'Doctorate']
income_order = ['Less than $40K', '$40K - $60K', '$60K - $80K',
                '$80K - $120K', '$120K +']
card_order = ['Blue', 'Silver', 'Gold', 'Platinum']

df['Education_Level'] = pd.Categorical(df['Education_Level'],
                                       categories=education_order, ordered=True)
df['Income_Category'] = pd.Categorical(df['Income_Category'],
                                       categories=income_order, ordered=True)
df['Card_Category'] = pd.Categorical(df['Card_Category'],
                                     categories=card_order, ordered=True)

print(f"{df.shape[0]:,} customers, {df.shape[1]} columns")
print(f"Overall churn rate: {df['Churn'].mean():.2%}")

10,127 customers, 20 columns
Overall churn rate: 16.07%


## Two helpers

`churn_rate_by` produces a churn rate per level of a variable, with the group
size beside it so that a rate resting on 20 customers is never read as though it
rested on 2,000.

`compare_groups` runs a Mann-Whitney U test for a numeric variable across the
two groups. It is used rather than a t-test because several of these
distributions are strongly skewed and one of them is mostly zeros.

In [5]:
def churn_rate_by(column, min_size=1):
    """Churn rate at each level of `column`, with group sizes."""
    grouped = df.groupby(column, observed=True)['Churn']
    out = pd.DataFrame({
        'customers': grouped.size(),
        'churn_rate_pct': (grouped.mean() * 100).round(1),
    })
    out['reliable'] = np.where(out['customers'] >= 30, '', 'small sample')
    return out[out['customers'] >= min_size]


def compare_groups(column):
    """Median in each group, Mann-Whitney p, and a rank-biserial effect size."""
    churned = df.loc[df['Churn'] == 1, column]
    retained = df.loc[df['Churn'] == 0, column]
    u_stat, p_value = mannwhitneyu(churned, retained)
    rank_biserial = 1 - 2 * u_stat / (len(churned) * len(retained))
    return {
        'variable': column,
        'median_churned': round(churned.median(), 3),
        'median_retained': round(retained.median(), 3),
        'p_value': p_value,
        'effect_size': round(rank_biserial, 3),
    }

**Reading the effect size.** Rank-biserial runs from -1 to +1 and answers a
plain question: pick one churned and one retained customer at random, how often
does the churned one score lower? 0 means the two groups are indistinguishable;
0.5 is a large separation. The sign says which way round it goes - positive
means churned customers score lower on that variable.

---

# Section 2: H1 - Disengagement precedes departure

> Customers who become less active are more likely to have churned.

In [6]:
churn_rate_by('Months_Inactive_12_mon')

,customers,churn_rate_pct,reliable
Months_Inactive_12_mon,,,
0,29,51.7,small sample
1,2233,4.5,
2,3282,15.4,
3,3846,21.5,
4,435,29.9,
5,178,18.0,
6,124,15.3,


The pattern holds across the bulk of the file but is **not** a clean straight
line, and the exceptions matter:

* From 1 to 4 months inactive the churn rate climbs steadily, 4.5% to 29.9%.
  This is the main effect and it covers 9,796 of the 10,127 customers.
* **0 months inactive shows 51.7% churn** - the highest rate in the table. That
  rests on 29 customers, which is too few to conclude anything from. It is more
  likely an artefact of how the field is recorded for closing accounts than a
  real finding, and it is not used.
* 5 and 6 months fall back to 18.0% and 15.3%, on 178 and 124 customers.

So the honest version of H1 is narrower than the hypothesis: inactivity tracks
churn over the range where nearly all customers sit, and the tails are unstable.

In [7]:
pd.DataFrame([compare_groups('Total_Trans_Ct'),
              compare_groups('Total_Trans_Amt'),
              compare_groups('Months_Inactive_12_mon')])

,variable,median_churned,median_retained,p_value,effect_size
0,Total_Trans_Ct,43.0,71.0,0.000000e+00,0.591
1,Total_Trans_Amt,2329.0,4100.0,2.719009e-112,0.352
2,Months_Inactive_12_mon,3.0,2.0,5.423116e-67,-0.256


**Transaction count is the strongest single signal in the dataset.**

The median churned customer made **43 transactions** against **71** for the
median retained customer - a 39% gap, with an effect size of 0.59. That is far
larger than anything demographic in section 6, and it is the finding the
retention team can act on.

**H1 is supported**, with the caveat about the inactivity tails above.

---

# Section 3: H2 - A thinner relationship is easier to leave

> The fewer products a customer holds, the lower the switching cost.

In [8]:
churn_rate_by('Total_Relationship_Count')

,customers,churn_rate_pct,reliable
Total_Relationship_Count,,,
1,910,25.6,
2,1243,27.8,
3,2305,17.4,
4,1912,11.8,
5,1891,12.0,
6,1866,10.5,


In [9]:
pd.DataFrame([compare_groups('Total_Relationship_Count')])

,variable,median_churned,median_retained,p_value,effect_size
0,Total_Relationship_Count,3.0,4.0,2.905717e-51,0.231


**H2 is supported**, and the shape is more useful than a simple correlation
would suggest.

Churn is **25.6%** at one product and **27.8%** at two, then drops sharply to
17.4% at three and settles around 10-12% from four products upward. It is not a
gentle gradient - it is a cliff between two products and three.

That has a direct operational reading: the customers worth approaching are those
holding one or two products, and the goal is getting them to a third. Beyond
four, additional products buy very little extra retention.

---

# Section 4: H3 - A falling spend trend is a warning sign

> Churned customers show a sharper quarter-on-quarter decline than retained
> ones.

In [10]:
pd.DataFrame([compare_groups('Total_Ct_Chng_Q4_Q1'),
              compare_groups('Total_Amt_Chng_Q4_Q1')])

,variable,median_churned,median_retained,p_value,effect_size
0,Total_Ct_Chng_Q4_Q1,0.531,0.721,1.908241e-216,0.491
1,Total_Amt_Chng_Q4_Q1,0.701,0.743,1.064474e-24,0.160


**H3 is supported for transaction count, and only weakly for amount.**

The change in *transaction count* separates the groups clearly: a median ratio of
**0.53 for churned customers against 0.72 for retained**, effect size 0.49. The
median churner roughly halved how often they used the card.

The change in *amount* barely separates them at all - 0.70 against 0.74, effect
size 0.16. Both are significant, but significance is cheap at this sample size
and the second difference is too small to act on.

This distinction is worth stating plainly, because the two variables sound
interchangeable and are not. **Customers stop using the card before they stop
spending on it.** Frequency drops away first; the amount per remaining
transaction holds up, which masks the change in any report that tracks spend
alone.

---

# Section 5: H4 - Repeated contact signals an unresolved problem

> Customers who contacted the bank more often are more likely to have churned.

In [11]:
churn_rate_by('Contacts_Count_12_mon')

,customers,churn_rate_pct,reliable
Contacts_Count_12_mon,,,
0,399,1.8,
1,1499,7.2,
2,3227,12.5,
3,3380,20.1,
4,1392,22.6,
5,176,33.5,
6,54,100.0,


**H4 is supported, and this is the sharpest gradient in the data.**

Churn rises monotonically at every step: 1.8% at zero contacts, 12.5% at two,
22.6% at four, 33.5% at five - and **100% at six contacts**.

That last row deserves care rather than excitement. All 54 customers who made
six contacts churned. Every one. A rate of exactly 100% on a clean threshold is
almost always a sign of something structural rather than a behavioural law - it
may be that six contacts is where an account closure process begins, in which
case the contacts are part of leaving rather than a warning that precedes it.

The direction of causation is genuinely unknown here, and it changes what the
finding is worth. If repeated contact frustrates customers into leaving, fixing
first-contact resolution reduces churn. If contacts are simply what leaving
looks like administratively, then acting on this signal at five or six contacts
is far too late to help. The safe reading is that contact count is a **useful
early flag at three to five**, where the gradient is steep but the outcome is
not yet settled.

In [12]:
pd.DataFrame([compare_groups('Contacts_Count_12_mon')])

,variable,median_churned,median_retained,p_value,effect_size
0,Contacts_Count_12_mon,3.0,2.0,1.110888e-80,-0.286


---

# Section 6: H5 - Behaviour beats demography

> Demographic attributes separate the two groups less sharply than behavioural
> ones.

This is the hypothesis that matters commercially, so it needs a comparison that
puts a six-level categorical variable and a continuous one on the same footing.

Two measures are used together:

* **Spread** - the gap in percentage points between the highest and lowest churn
  rate across a variable's levels, with continuous variables cut into quintiles
  and any group under 30 customers excluded. This is the plain-language version:
  how much does knowing this attribute move the churn rate?
* **Cramer's V** - a standard 0-to-1 measure of association strength, which does
  not depend on the number of levels.

In [13]:
def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.values.sum()
    rows, cols = contingency.shape
    return np.sqrt((chi2 / n) / max(1, min(rows, cols) - 1))


demographic = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
               'Card_Category', 'Customer_Age', 'Dependent_count']
behavioural = ['Total_Trans_Ct', 'Total_Trans_Amt', 'Total_Ct_Chng_Q4_Q1',
               'Total_Amt_Chng_Q4_Q1', 'Total_Revolving_Bal',
               'Avg_Utilization_Ratio', 'Contacts_Count_12_mon',
               'Months_Inactive_12_mon', 'Total_Relationship_Count',
               'Credit_Limit']

rows = []
for col in demographic + behavioural:
    series = df[col]
    if series.dtype.kind in 'if' and series.nunique() > 8:
        binned = pd.qcut(series, 5, duplicates='drop').astype(str)
    else:
        binned = series.astype('object')

    recorded = binned.notna() & series.notna()
    grouped = df.loc[recorded].groupby(binned[recorded])['Churn'].agg(['mean', 'size'])
    grouped = grouped[grouped['size'] >= 30]

    rows.append({
        'variable': col,
        'type': 'demographic' if col in demographic else 'behavioural',
        'spread_pp': round((grouped['mean'].max() - grouped['mean'].min()) * 100, 1),
        'cramers_v': round(cramers_v(binned[recorded], df.loc[recorded, 'Churn']), 3),
        'customers_used': int(recorded.sum()),
    })

ranking = pd.DataFrame(rows).sort_values('spread_pp', ascending=False).reset_index(drop=True)
ranking

,variable,type,spread_pp,cramers_v,customers_used
0,Contacts_Count_12_mon,behavioural,98.2,0.241,10127
1,Total_Trans_Amt,behavioural,43.1,0.427,10127
2,Total_Ct_Chng_Q4_Q1,behavioural,35.0,0.365,10127
3,Total_Trans_Ct,behavioural,33.7,0.405,10127
4,Months_Inactive_12_mon,behavioural,25.4,0.198,10127
5,Total_Revolving_Bal,behavioural,24.5,0.300,10127
6,Avg_Utilization_Ratio,behavioural,17.9,0.230,10127
7,Total_Relationship_Count,behavioural,17.3,0.167,10127
8,Total_Amt_Chng_Q4_Q1,behavioural,15.2,0.155,10127
9,Credit_Limit,behavioural,11.7,0.105,10127


**H5 is supported, and the separation is total.**

Every single behavioural variable outranks every single demographic one. There
is no overlap between the two groups at all:

| | Spread | Cramer's V |
|---|---|---|
| Behavioural (10 variables) | 11.7 - 98.2 pp | 0.105 - 0.427 |
| Demographic (7 variables) | 1.8 - 5.9 pp | 0.015 - 0.038 |

The weakest behavioural variable, `Credit_Limit`, moves the churn rate twice as
far as the strongest demographic one.

To put the demographic numbers in context: the entire spread of churn across
education levels is 5.9 percentage points, and across marital status 1.8. On
`Customer_Age` the Mann-Whitney test does not even reach significance
(p = 0.078) despite 10,127 rows - the median churned customer is 47 and the
median retained customer is 46.

In [14]:
pd.DataFrame([compare_groups('Customer_Age'),
              compare_groups('Credit_Limit')])

,variable,median_churned,median_retained,p_value,effect_size
0,Customer_Age,47.0,46.0,7.809995e-02,-0.028
1,Credit_Limit,4178.0,4643.5,3.007841e-07,0.080


### What this means for how the bank should act

This is the result with the clearest consequence, and it happens to point the
same way as the ethical constraint in the README.

Targeting retention offers on demographics would carry **discrimination risk**
for **almost no return**. Gender and age are protected characteristics under the
Equality Act 2010; income, education and marital status are not protected in
themselves, but they correlate with characteristics that are, so they can act as
proxies. And all of them barely move the churn rate.
Behavioural triggers are the better choice on both counts at once. That is a
convenient result, and it is worth saying that it did not have to come out this
way: had education been strongly predictive, the recommendation would still have
been to avoid targeting on it, and the analysis would have been less comfortable.

One demographic figure invites misreading and should be handled carefully:
Platinum cardholders show 25% churn against 16% overall. There are **20 Platinum
customers in the file**. That row is excluded from the ranking above by the
30-customer floor, and no conclusion should be drawn from it.

In [15]:
churn_rate_by('Card_Category')

,customers,churn_rate_pct,reliable
Card_Category,,,
Blue,9436,16.1,
Silver,555,14.8,
Gold,116,18.1,
Platinum,20,25.0,small sample


---

# Section 7: An unhypothesised finding - the dormant balance

The strongest signal that none of the five hypotheses anticipated concerns the
revolving balance.

In [16]:
df['zero_balance'] = (df['Total_Revolving_Bal'] == 0).astype(int)

pd.DataFrame({
    'share_with_zero_balance': df.groupby('Attrition_Flag')['zero_balance'].mean().round(4),
    'customers': df.groupby('Attrition_Flag')['zero_balance'].size(),
})

,share_with_zero_balance,customers
Attrition_Flag,,
Attrited Customer,0.5489,1627
Existing Customer,0.1855,8500


In [17]:
pd.DataFrame([compare_groups('Total_Revolving_Bal'),
              compare_groups('Avg_Utilization_Ratio')])

,variable,median_churned,median_retained,p_value,effect_size
0,Total_Revolving_Bal,0.0,1364.000,1.915679e-129,0.375
1,Avg_Utilization_Ratio,0.0,0.211,2.869097e-129,0.375


**The median churned customer carries a revolving balance of zero.**

54.9% of churned customers sit at exactly zero, against 18.6% of retained
customers - three times the rate. The same shows in utilisation: a median of
0.00 for churned customers against 0.21 for retained.

The interpretation needs care, because zero has two quite different meanings
here. A customer who pays their balance in full every month is a good customer
with a zero balance. A customer who has quietly stopped using the card also has
a zero balance. Both look identical in this column.

What separates them is section 2: churned customers also make 43 transactions to
the retained customer's 71. A zero balance *combined with* falling transaction
count is the dormancy pattern. A zero balance on its own is not - and a retention
campaign aimed at everyone with a zero balance would spend most of its budget on
customers who are simply paying their bills.

---

# Section 8: Save the ranking

In [18]:
os.makedirs('outputs/reports', exist_ok=True)
ranking.to_csv('outputs/reports/variable_ranking.csv', index=False)
print("Saved to outputs/reports/variable_ranking.csv")

Saved to outputs/reports/variable_ranking.csv


### A method that did not work

`ppscore` is in the project requirements and was tried as a way of scoring every
variable against the target on a single scale, which would have been neater than
the two measures used above. It is run below so the result can be seen rather
than described.

In [19]:
import ppscore as pps

pps_scores = pps.predictors(df.drop(columns=['Attrition_Flag', 'zero_balance']),
                            'Churn')[['x', 'ppscore']]
pps_scores.reset_index(drop=True)

G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packag

G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packages\ppscore\calculation.py:201: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  or is_categorical_dtype(series)
G:\Projects\Credit_Card_Churn-Analysis-main\Credit_Card_Churn-Analysis-main\.venv\Lib\site-packag

,x,ppscore
0,Customer_Age,0
1,Gender,0
2,Dependent_count,0
3,Education_Level,0
4,Marital_Status,0
5,Income_Category,0
6,Card_Category,0
7,Months_on_book,0
8,Total_Relationship_Count,0
9,Months_Inactive_12_mon,0


In [20]:
print(f"Variables scored:      {len(pps_scores)}")
print(f"Variables scoring 0.00: {(pps_scores['ppscore'] == 0).sum()}")
print(f"Highest score:          {pps_scores['ppscore'].max():.4f}")

Variables scored:      18
Variables scoring 0.00: 18
Highest score:          0.0000


It returns **0.00 for all 18 variables**, including transaction count.

That is not a finding about the data - it is a property of the measure. PPS
scores a variable against a naive baseline using weighted F1, and on a target
that is 84% one class, the naive baseline is already strong enough that no single
variable beats it. The measure is unsuitable for an imbalanced binary target and
was replaced with the spread and Cramer's V approach above.

It is recorded here rather than quietly dropped, because a table of eighteen
zeros is the kind of output that gets mistaken for evidence that nothing
predicts churn.

---

# Conclusions and Next Steps

**Verdict on the five hypotheses:**

| | Hypothesis | Verdict |
|---|---|---|
| H1 | Disengagement precedes departure | **Supported** - transaction count is the strongest single signal (43 vs 71 median, effect size 0.59). Inactivity tracks churn from 1-4 months; the tails are unstable and unused. |
| H2 | A thinner relationship is easier to leave | **Supported** - 25-28% churn at one or two products against 10-12% at four or more. A cliff, not a gradient. |
| H3 | A falling spend trend is a warning sign | **Partly supported** - clear for transaction *count* (0.53 vs 0.72), weak for *amount* (0.70 vs 0.74). |
| H4 | Repeated contact signals a problem | **Supported** - monotonic from 1.8% at zero contacts to 100% at six. Direction of causation unknown; useful as a flag at 3-5, too late at 6. |
| H5 | Behaviour beats demography | **Supported without exception** - every behavioural variable outranks every demographic one on both measures. |

**Answering BR2:** the attributes that distinguish churned from retained
customers are, in order, transaction count, transaction amount, the
quarter-on-quarter change in transaction count, contact frequency, revolving
balance and product holdings. Demographic attributes do not meaningfully
separate the two groups.

**The one finding not in the hypotheses:** 54.9% of churned customers carry a
zero revolving balance against 18.6% of retained ones - but zero balance alone
does not identify a churner, since it also describes customers who pay in full.
It is only informative alongside falling transaction count.

**Carried caveats:** the 100% churn rate at six contacts is a threshold effect of
unknown direction; the 51.7% at zero months inactive rests on 29 customers; the
25% Platinum churn rate rests on 20; and `Months_on_book` remains unusable for
the reasons in notebook 02.

Next: notebook 04, turning the findings above into the charts BR3 asks for.